# Titanic survival — 1-minute EDA

Source: `Titanic.csv` · 891 passengers · target: `Survived` (0/1).

**Main takeaway:** sex and ticket class dominate survival. Women in 1st class survived at 96.8%; men in 3rd class at 13.5%.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("Titanic.csv")
df["FamilySize"] = df["SibSp"] + df["Parch"]
df["Title"] = (
    df["Name"].str.extract(r",\s*([^\.]+)\.")[0]
    .replace({"Ms": "Miss", "Mlle": "Miss", "Mme": "Mrs"})
)

print(df.shape)
df.head()

## Missing values and overall rate

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing)
print()
print(f"Overall survival: {df['Survived'].mean():.1%}  ({df['Survived'].sum()} / {len(df)})")

## Survival by sex and class

In [ ]:
sex = df.groupby("Sex")["Survived"].agg(["mean", "sum", "count"])
pclass = df.groupby("Pclass")["Survived"].agg(["mean", "sum", "count"])
cross = df.groupby(["Sex", "Pclass"])["Survived"].agg(["mean", "sum", "count"])

display(sex.style.format({"mean": "{:.1%}"}))
display(pclass.style.format({"mean": "{:.1%}"}))
display(cross.style.format({"mean": "{:.1%}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

df["Survived"].value_counts().sort_index().plot(
    kind="pie", ax=axes[0], autopct="%1.1f%%",
    labels=["Died (549)", "Survived (342)"], colors=["#c44", "#3a7"],
)
axes[0].set_ylabel("")
axes[0].set_title("Overall outcome")

pclass["mean"].mul(100).plot(kind="bar", ax=axes[1], color="#3a6ea8", rot=0)
axes[1].set_ylabel("Survival rate (%)")
axes[1].set_xlabel("Passenger class")
axes[1].set_title("Survival by class")
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
rates = (
    df.groupby(["Sex", "Pclass"])["Survived"].mean().mul(100)
    .rename("rate").reset_index()
)
rates["label"] = rates["Sex"].str.capitalize() + " · class " + rates["Pclass"].astype(str)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(rates["label"], rates["rate"], color="#3a7")
ax.set_xlabel("Survival rate (%)")
ax.set_title("Survival by sex × class")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

## Age, family size, fare, titles

In [ ]:
known = df.dropna(subset=["Age"]).copy()
known["AgeBin"] = pd.cut(known["Age"], bins=[-0.1, 12, 18, 60, 120], labels=["0–12", "13–18", "19–60", "60+"])
age = known.groupby("AgeBin", observed=False)["Survived"].agg(["mean", "count"])

fam = df.assign(
    FamBin=pd.cut(df["FamilySize"], bins=[-0.1, 0, 3, 20], labels=["Alone (0)", "Small (1–3)", "Large (4+)"])
).groupby("FamBin", observed=False)["Survived"].agg(["mean", "count"])

display(age.style.format({"mean": "{:.1%}"}))
display(fam.style.format({"mean": "{:.1%}"}))
print("Median fare — survived:", df.loc[df.Survived == 1, "Fare"].median())
print("Median fare — died:    ", df.loc[df.Survived == 0, "Fare"].median())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

age["mean"].mul(100).plot(kind="bar", ax=axes[0], color="#3a6ea8", rot=0)
axes[0].set_ylabel("Survival rate (%)")
axes[0].set_xlabel("Age (known only, n=714)")
axes[0].set_title("Survival by age")
axes[0].set_ylim(0, 100)

fam["mean"].mul(100).plot(kind="bar", ax=axes[1], color="#c90", rot=0)
axes[1].set_ylabel("Survival rate (%)")
axes[1].set_xlabel("Family size (SibSp + Parch)")
axes[1].set_title("Survival by family size")
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
seg = pd.concat([
    pd.DataFrame({"n": [len(df)], "survival": [df.Survived.mean()]}, index=["All passengers"]),
    df.groupby("Sex")["Survived"].agg(n="count", survival="mean"),
    df.groupby("Embarked")["Survived"].agg(n="count", survival="mean").rename(index=lambda x: f"Embarked {x}"),
    pd.DataFrame({
        "n": [df.Cabin.notna().sum(), df.Cabin.isna().sum()],
        "survival": [
            df.loc[df.Cabin.notna(), "Survived"].mean(),
            df.loc[df.Cabin.isna(), "Survived"].mean(),
        ],
    }, index=["Cabin recorded", "Cabin missing"]),
    df[df.Title.isin(["Mrs", "Miss", "Master", "Mr"])]
      .groupby("Title")["Survived"].agg(n="count", survival="mean")
      .rename(index=lambda t: f"Title {t}"),
])
seg.style.format({"n": "{:.0f}", "survival": "{:.1%}"})